# ML-06 — Signal Audit: Do the Flags Hold?

Runs **offline** on the bundled anonymized slice (`data/raw/content_refresh_anonymized.csv`,
30,000 pages) — no warehouse token needed. The full-release version of the same signal checks is
in `w04_baseline_score.ipynb` (March 2026 → April 2026 decline label).

> Label caveat up front: on this slice the only available label is the **same-window proxy**
> `is_declining_label = (trend_direction == 'down')`. It is a *defined rule*, not a future
> outcome, so every verdict below is an **observed, in-window association** — decision-support
> language only.

## 1. Distributions

Quick shape of the four signals the audit leans on — before any flag is trusted, look at the
spread and the missing values.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
base = df["is_declining_label"].mean()

cols = ["impressions_90d", "ctr", "avg_position", "days_since_last_update", "word_count", "trend_pct"]
desc = df[cols].describe().T[["count", "mean", "50%", "max"]].round(3)
desc["missing"] = len(df) - desc["count"]
print(f"rows: {len(df):,}  |  proxy decline base rate: {base:.3f}")
print(desc.to_string())

rows: 30,000  |  proxy decline base rate: 0.542
                          count      mean      50%       max  missing
impressions_90d         30000.0  5200.366   731.00  517715.0      0.0
ctr                     30000.0     0.511     0.07     100.0      0.0
avg_position            30000.0    16.342    10.80     245.0      0.0
days_since_last_update  30000.0    46.098    20.00     373.0      0.0
word_count              22301.0  3107.760  2877.00    9546.0   7699.0
trend_pct               26612.0    -4.786   -33.50   44900.0   3388.0


**Read:** `ctr` is heavily skewed (median 0.07, max 100), `avg_position` includes `0` (no
position data), and `trend_pct` is missing for ~11% of pages. Any flag built on these must state
how it treats zero/missing — the audit below does.

## 2. Signal test #1 / #2 / #3 (verdict each)

Three transparent signals, each tested the same way: bucket the pool, print **n** next to the
decline rate, and write a verdict. A signal with no lift is reported as no lift.

In [2]:
# ---- Signal test #1: VOLUME. Does demand gate, or does it drive, decline risk? ----
tiers = pd.cut(df["impressions_90d"], [0, 1_000, 10_000, np.inf], labels=["low (100-1k)", "moderate (1k-10k)", "high (10k+)"])
sigA = df.groupby(tiers, observed=True).agg(n=("content_id", "count"), decline_rate=("is_declining_label", "mean"))
sigA = sigA.round(3)
print("SIGNAL CHECK 1 - volume")
print(sigA.to_string())
print()
print("VERDICT: MIXED - volume GATES the pool (the buckets are very uneven in n) but it is not a")
print("clean risk driver: the middle bucket is highest, so 'bigger = riskier' is FALSE on this slice.")
print("Interpretation: use volume for impact/exposure, never as the decline score itself.")

SIGNAL CHECK 1 - volume
                       n  decline_rate
impressions_90d                       
low (100-1k)       16491         0.499
moderate (1k-10k)   9907         0.620
high (10k+)         3602         0.524

VERDICT: MIXED - volume GATES the pool (the buckets are very uneven in n) but it is not a
clean risk driver: the middle bucket is highest, so 'bigger = riskier' is FALSE on this slice.
Interpretation: use volume for impact/exposure, never as the decline score itself.


In [3]:
# ---- Signal test #2: CTR vs POSITION. The CTR-for-position edge. ----
vis = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()
vis["band"] = np.select(
    [vis["avg_position"].le(3), vis["avg_position"].le(10), vis["avg_position"].le(20)],
    ["top3", "p1", "p2"], default="deep")
vis["ctr_tercile"] = vis.groupby("band")["ctr"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 3, labels=["bottom", "mid", "top"]))
sigB = vis.groupby(["band", "ctr_tercile"], observed=True).agg(
    n=("content_id", "count"), decline_rate=("is_declining_label", "mean")).round(3)
print("SIGNAL CHECK 2 - CTR vs position (n printed per cell)")
print(sigB.to_string())
top3 = sigB.loc["top3"]
print()
print(f"VERDICT: CONFIRMED for page-one pages - top3 bottom-ctr declines at {top3.loc['bottom','decline_rate']:.3f}")
print(f"vs {top3.loc['top','decline_rate']:.3f} for the top third, and p1 shows the same ordering.")
print("The edge fades deeper in the results. This is the signal a CTR-for-position rule can lean on.")

SIGNAL CHECK 2 - CTR vs position (n printed per cell)
                     n  decline_rate
band ctr_tercile                    
deep bottom       1568         0.546
     mid          1567         0.657
     top          1568         0.561
p1   bottom       2361         0.683
     mid          2361         0.567
     top          2362         0.486
p2   bottom       1486         0.686
     mid          1486         0.595
     top          1487         0.564
top3 bottom        160         0.881
     mid           160         0.881
     top           160         0.450

VERDICT: CONFIRMED for page-one pages - top3 bottom-ctr declines at 0.881
vs 0.450 for the top third, and p1 shows the same ordering.
The edge fades deeper in the results. This is the signal a CTR-for-position rule can lean on.


In [4]:
# ---- Signal test #3: FRESHNESS. Does 'not updated lately' predict decline here? ----
df["fresh_band"] = pd.cut(df["days_since_last_update"], [-1, 90, 180, np.inf], labels=["<90d", "90-180d", "180d+"])
sigC = df.groupby("fresh_band", observed=True).agg(
    n=("content_id", "count"), decline_rate=("is_declining_label", "mean"),
    median_impressions=("impressions_90d", "median")).round(3)
print("SIGNAL CHECK 3 - freshness")
print(sigC.to_string())
print()
print("VERDICT: WEAK / NON-MONOTONE - the 90-180d bucket is highest, the 180d+ bucket has only")
print(f"{int(sigC.loc['180d+','n'])} pages, so freshness is not a reliable standalone decline flag on this slice.")
print("It stays a *review* cue (a stale visible page is worth a look), not a risk score.")

SIGNAL CHECK 3 - freshness
                n  decline_rate  median_impressions
fresh_band                                         
<90d        20655         0.512               472.0
90-180d      9171         0.611              1692.0
180d+         174         0.471                15.5

VERDICT: WEAK / NON-MONOTONE - the 90-180d bucket is highest, the 180d+ bucket has only
174 pages, so freshness is not a reliable standalone decline flag on this slice.
It stays a *review* cue (a stale visible page is worth a look), not a risk score.


## 3. The flag-linked test

The starter pipeline's own reason flags, tested directly against the proxy label. A flag earns its
place only if the decline rate **inside** the flag is clearly above the base rate **outside** it.

In [5]:
flags = {
    "low_ctr_visible_page": (df["impressions_90d"] >= 500) & (df["avg_position"].between(0.001, 20)) & (df["ctr"] < 0.5),
    "thin_visible_page": (df["word_count"].between(0.001, 1200)) & (df["impressions_90d"] >= 250),
    "stale_visible_page": (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500),
}
rows = []
for name, mask in flags.items():
    rows.append({"flag": name, "n": int(mask.sum()),
                 "decline_if_flagged": round(df.loc[mask, "is_declining_label"].mean(), 3),
                 "decline_if_not": round(df.loc[~mask, "is_declining_label"].mean(), 3),
                 "lift": round(df.loc[mask, "is_declining_label"].mean() - df.loc[~mask, "is_declining_label"].mean(), 3)})
flag_tbl = pd.DataFrame(rows)
print(f"base rate: {base:.3f}")
print(flag_tbl.to_string(index=False))
print()
print("VERDICT: low_ctr_visible_page earns its place (a clear lift on ~9.8k pages). thin_visible_page")
print("shows no lift here. stale_visible_page has a huge lift but only 17 pages - too thin to trust.")

base rate: 0.542
                flag    n  decline_if_flagged  decline_if_not   lift
low_ctr_visible_page 9759               0.627           0.501  0.126
   thin_visible_page   83               0.530           0.542 -0.012
  stale_visible_page   17               0.941           0.542  0.399

VERDICT: low_ctr_visible_page earns its place (a clear lift on ~9.8k pages). thin_visible_page
shows no lift here. stale_visible_page has a huge lift but only 17 pages - too thin to trust.


## 4. What this means in practice

- **Keep:** CTR-vs-position (the one confirmed edge) and `low_ctr_visible_page` as a review flag.
- **Gate, don't score:** volume sets exposure/impact; it is not a risk signal.
- **Demote:** freshness alone and `thin_visible_page` — useful context, weak standalone predictors here.
- **Honest word:** all of the above is an in-window association under a proxy label. The
  full-release check (`w04_baseline_score.ipynb`) re-runs these tests against a genuine
  March→April decline label; there, `low_ctr_high_position` and volume-as-gate both hold.

In [6]:
metrics = {
    "task": "ml-06",
    "data": "data/raw/content_refresh_anonymized.csv",
    "rows": int(len(df)),
    "label": "is_declining_label = (trend_direction == 'down') [same-window proxy]",
    "base_rate": round(float(base), 3),
    "signal_1_volume": {"verdict": "MIXED (gate, not driver)",
                        "decline_by_tier": sigA["decline_rate"].round(3).to_dict()},
    "signal_2_ctr_vs_position": {"verdict": "CONFIRMED (page-one)",
                                 "top3_bottom_ctr": round(float(sigB.loc[("top3", "bottom"), "decline_rate"]), 3),
                                 "top3_top_ctr": round(float(sigB.loc[("top3", "top"), "decline_rate"]), 3)},
    "signal_3_freshness": {"verdict": "WEAK / non-monotone",
                           "decline_by_band": sigC["decline_rate"].round(3).to_dict()},
    "flag_lift": flag_tbl.to_dict(orient="records"),
}
(OUT / "w04_signal_audit_metrics.json").write_text(json.dumps(metrics, indent=2))
print("metrics receipt written:", OUT / "w04_signal_audit_metrics.json")

metrics receipt written: C:\Users\Ananya\OneDrive\文件\FlyRank Intern\flyrank-ml-internship-start\work\outputs\w04_signal_audit_metrics.json


### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed offline on the bundled slice)
- [x] No client names, URLs, or private queries anywhere — aggregate buckets only
- [x] Verdicts use careful words: observed / measured / proxy label / decision-support
- [x] Committed to my repo under `work/notebooks/`